In [382]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
import numpy as np

n_samples = 1000
X, y = make_classification(n_samples=n_samples, n_features=5, n_informative=2, n_redundant=3, random_state=7)

n_estimators = 200
rf = RandomForestClassifier(n_estimators=n_estimators, bootstrap=True, random_state=42)
rf.fit(X, y)


X

array([[ 1.23416248, -0.46953005, -1.13255839, -0.28584845,  0.65657723],
       [-0.10172121,  0.52630421,  1.11327977,  0.22839354, -2.34697333],
       [-1.67042565,  0.37593003,  0.98994988,  0.27785078,  0.33191994],
       ...,
       [-1.22128582,  0.18846993,  0.54308985,  0.16685596,  0.64886212],
       [-0.63486218,  0.11766954,  0.32351559,  0.09501124,  0.24467755],
       [-1.12064531, -0.16934424, -0.21762425,  0.00931946,  2.20490568]])

In [324]:
x_pred = [-0.42016338,  0.26413616, -1.64544487, -0.70228821 , 0.51820725]
theta_is = []
for ii in range(n_samples):
    indices_without_zero = []
    for i, sample in enumerate(rf.estimators_samples_):
        if ii not in sample:
            indices_without_zero.append(i)
    if len(indices_without_zero) == 0 or len(indices_without_zero) == n_estimators:
        continue
    theta_is.append(np.array([rf.estimators_[i].predict_proba([x_pred])[0,1] for i in indices_without_zero  ]).mean(axis=0))

theta = rf.predict_proba([x_pred])
var_jka_biased = np.sum((theta_is- theta[0,1])**2) * (n_samples-1)/n_samples

tree_preds = np.array([estimator.predict_proba([x_pred])[0,1] for estimator in rf.estimators_])
var_jka_correction = (np.exp(1) - 1) * (n_samples/n_estimators) * np.var(tree_preds)
var_jka_unbiased = var_jka_biased - var_jka_correction

var_jka_unbiased


0.038439540235959635

In [383]:
#### YEyyyy ####
# Input prediction sample
x_pred = np.array([-0.42016338,  0.26413616, -1.64544487, -0.70228821 , 0.51820725]).reshape(1, -1)


# Precompute predictions for all trees
tree_preds = np.array([estimator.predict_proba(x_pred)[0, 1] for estimator in rf.estimators_])

# Cache the estimators' samples array for efficient reuse
estimators_samples = np.array(rf.estimators_samples_, dtype=object)

# Prepare a boolean mask for each sample's presence in each estimator's bootstrap
presence_mask = np.zeros((n_samples, n_estimators), dtype=bool)
for i, samples in enumerate(estimators_samples):
    # Convert the sample list to a NumPy array of integers
    samples = np.array(samples, dtype=int)
    presence_mask[samples, i] = True

# Calculate the theta_is efficiently
theta_is = []
for ii in range(n_samples):
    # Use the presence mask to identify trees where sample ii is not in the bootstrap sample
    indices_without_ii = np.where(~presence_mask[ii])[0]

    # Skip samples that are included in all or none of the trees
    if 0 < len(indices_without_ii) < n_estimators:
        # Calculate the mean prediction for these trees
        theta_is.append(tree_preds[indices_without_ii].mean())

# Convert theta_is to a NumPy array
theta_is = np.array(theta_is)

# Compute the final variance estimates
theta = rf.predict_proba(x_pred)
var_jka_biased = np.sum((theta_is - theta[0, 1]) ** 2) * (n_samples - 1) / n_samples

var_jka_correction = (np.exp(1) - 1) * (n_samples / n_estimators) * np.var(tree_preds)
var_jka_unbiased = var_jka_biased - var_jka_correction

var_jka_unbiased


0.038439540235959635

In [377]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
import time

# Generate a synthetic dataset
n_samples = 10000
n_features = 20
X, y = make_classification(n_samples=n_samples, n_features=n_features, n_informative=10, n_redundant=10, random_state=7)

# Convert X to a DataFrame
X_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])

# Initialize the RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, bootstrap=True, random_state=42)

# Benchmark with NumPy array
start_time = time.time()
rf.fit(X, y)
np_fit_time = time.time() - start_time

start_time = time.time()
np_pred = rf.predict(X)
np_pred_time = time.time() - start_time



In [379]:
X_df

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19
0,-3.365541,0.454614,-1.280969,-1.364312,1.091634,0.643096,-1.630384,-0.906170,-0.039504,3.513609,-2.256759,-4.800636,-3.463932,-3.286668,-1.282973,-2.880296,-1.306566,0.041394,-3.446461,0.351785
1,0.468355,0.501294,0.137403,-3.785282,3.119732,0.836822,-2.573819,-0.714994,-1.533857,-3.638781,-1.738510,-1.380142,-1.214173,-2.386068,-0.007058,-0.120215,-6.614866,-0.053519,-4.569494,7.392968
2,-0.670842,-4.442109,-3.662154,-2.089754,2.481423,-2.393701,-2.031174,3.956832,-4.177737,-3.719181,0.405496,0.371104,-4.110959,0.036245,-0.230803,2.768129,-0.674654,-0.121568,-2.438405,-0.441474
3,-0.437905,-4.578575,-2.873937,-2.791895,3.730298,-1.317075,-1.784370,1.303957,2.108059,1.034698,-2.503829,-0.585963,-4.630848,-1.077497,-3.117018,-1.894086,-1.559709,-1.889185,-4.904435,0.933411
4,-0.335294,-0.975333,-2.098818,3.142914,-2.952130,0.232579,-0.251556,1.720663,-1.665494,2.417439,3.041801,-0.538365,0.432972,-0.400665,1.281258,-0.617977,2.671766,-0.537900,0.233603,-5.025167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.455095,1.351146,2.683705,0.130665,-3.639204,-0.426301,0.365875,1.474679,-3.538505,-1.038979,2.333246,-0.327792,-1.646559,-0.408687,4.072082,-0.216688,-1.841971,2.968382,2.591615,-1.204613
9996,0.141668,2.576758,7.108985,-1.586332,-2.165284,0.891903,0.013807,-0.499968,-1.651561,0.232739,0.037545,1.641015,-0.589400,0.858686,1.781060,-2.192936,-3.781365,3.956668,4.352434,1.847640
9997,-0.891116,-3.754633,-3.937868,0.958200,0.576656,0.310377,1.148179,2.744106,3.295895,7.562070,-3.564971,-0.823694,-3.085498,3.723425,-5.481531,-2.033595,4.762567,-0.336635,-1.135485,-8.161188
9998,-0.787487,-3.133295,-0.968790,1.006064,-0.023451,-2.355712,-0.163734,0.162831,2.706117,1.522561,1.296094,0.958868,-0.145755,1.267814,-0.429962,-0.407137,4.646744,-1.163705,1.653174,-4.156258


In [378]:
X

array([[-3.36554149,  0.4546143 , -1.28096872, ...,  0.04139398,
        -3.44646108,  0.3517846 ],
       [ 0.46835487,  0.50129357,  0.13740326, ..., -0.05351938,
        -4.56949358,  7.39296759],
       [-0.67084224, -4.44210945, -3.66215448, ..., -0.12156833,
        -2.43840548, -0.44147394],
       ...,
       [-0.891116  , -3.75463319, -3.93786771, ..., -0.33663487,
        -1.13548518, -8.16118766],
       [-0.7874867 , -3.13329544, -0.96878951, ..., -1.1637052 ,
         1.65317355, -4.15625766],
       [-2.38820102, -3.00482239, -2.64026697, ..., -5.05416364,
         0.83485303, -4.15857541]])

In [380]:

# Benchmark with DataFrame
rf = RandomForestClassifier(n_estimators=200, bootstrap=True, random_state=42)  # Re-initialize to avoid effects of cached results

start_time = time.time()
rf.fit(X_df, y)
df_fit_time = time.time() - start_time

start_time = time.time()
df_pred = rf.predict(X)
df_pred_time = time.time() - start_time

# Results
print(f"NumPy array fit time: {np_fit_time:.4f} seconds")
print(f"NumPy array prediction time: {np_pred_time:.4f} seconds")
print(f"DataFrame fit time: {df_fit_time:.4f} seconds")
print(f"DataFrame prediction time: {df_pred_time:.4f} seconds")


NumPy array fit time: 5.5060 seconds
NumPy array prediction time: 0.1475 seconds
DataFrame fit time: 5.9105 seconds
DataFrame prediction time: 0.1494 seconds


c:\Users\rehan\anaconda3\envs\ma-thesis\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
